## Creating a Deep Reinforcement earning model to identify deepFake videos


# Dataset
importing the FaceaForensics++ dataset from kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("xdxd003/ff-c23")

print("Path to dataset files:", path)

 13%|█▎        | 2.10G/16.7G [00:55<06:25, 40.6MB/s]


KeyboardInterrupt: 

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("reubensuju/celeb-df-v2")

print("Path to dataset files:", path)

100%|██████████| 9.29G/9.29G [01:22<00:00, 120MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/reubensuju/celeb-df-v2/versions/1


Installing nessaccery libraries

In [79]:
import sys
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio timm stable-baselines3 facenet-pytorch

!{sys.executable} -m pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 timm stable-baselines3==2.1.0 facenet-pytorch

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.24.0
Uninstalling torchvision-0.24.0:
  Successfully uninstalled torchvision-0.24.0
Found existing installation: torchaudio 2.9.0
Uninstalling torchaudio-2.9.0:
  Successfully uninstalled torchaudio-2.9.0
Found existing installation: timm 1.0.27
Uninstalling timm-1.0.27:
  Successfully uninstalled timm-1.0.27
Found existing installation: stable-baselines3 2.1.0
Uninstalling stable-baselines3-2.1.0:
  Successfully uninstalled stable-baselines3-2.1.0
Found existing installation: facenet-pytorch 2.5.3
Uninstalling facenet-pytorch-2.5.3:
  Successfully uninstalled facenet-pytorch-2.5.3
  Using cached torch-2.9.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached torchvision-0.24.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
  Using cached torchaudio-2.9.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  

Importing nessaccery libraries

In [1]:
from timm import create_model
from facenet_pytorch import MTCNN
import os
import torch
import torch.nn as nn
from torchvision import transforms
import cv2
from PIL import Image
from tqdm import tqdm
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
import random

#for reinforcement learning agent and envronment
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

import cv2
import torch
import numpy as np
from torchvision import transforms
from facenet_pytorch import MTCNN
from PIL import Image

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Feature Extraction
 using swin-transformers from pytorch timm

Load Swin-Tiny for Feature Extraction

In [2]:
swin_tiny_model = create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes = 0)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

swin_tiny_model.eval()
swin_tiny_model.to(device)

mtcnn = MTCNN(keep_all = False, device=device, margin = 30)


In [4]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

'''
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
import torch

# This is the "Secret Sauce" to make your model work on real-world videos.
# It simulates the destruction caused by WhatsApp, TikTok, and Twitter compression.
def get_social_media_augmentations():
    """
    Returns an Albumentations pipeline designed to destroy camera/compression bias,
    forcing the model to rely ONLY on facial blending and temporal artifacts.
    """
    return A.Compose([
        # 1. Simulate bad internet / social media compression
        A.ImageCompression(quality_lower=40, quality_upper=90, p=0.7),

        # 2. Simulate bad webcams / low light noise
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
        A.ISONoise(color_shift=0.01, intensity=(0.1, 0.5), p=0.3),

        # 3. Simulate out-of-focus cameras or motion blur
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=(3, 7)),
        ], p=0.4),

        # 4. Color and lighting variations (simulating different phones/environments)
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.6),

        # 5. The standard resize and normalize for Swin Transformer
        A.Resize(224, 224),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
'''

'\nimport cv2\nimport albumentations as A\nfrom albumentations.pytorch import ToTensorV2\nfrom PIL import Image\nimport numpy as np\nimport torch\n\n# This is the "Secret Sauce" to make your model work on real-world videos.\n# It simulates the destruction caused by WhatsApp, TikTok, and Twitter compression.\ndef get_social_media_augmentations():\n    """\n    Returns an Albumentations pipeline designed to destroy camera/compression bias,\n    forcing the model to rely ONLY on facial blending and temporal artifacts.\n    """\n    return A.Compose([\n        # 1. Simulate bad internet / social media compression\n        A.ImageCompression(quality_lower=40, quality_upper=90, p=0.7),\n\n        # 2. Simulate bad webcams / low light noise\n        A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),\n        A.ISONoise(color_shift=0.01, intensity=(0.1, 0.5), p=0.3),\n\n        # 3. Simulate out-of-focus cameras or motion blur\n        A.OneOf([\n            A.GaussianBlur(blur_limit=(3, 7)),\n     

Video Frame Extraction

In [5]:
def get_video_features_mtcnn(video_path, max_frames=32):
    """
    Reads a video, detects faces, crops them, and extracts Swin features.
    """
    cap = cv2.VideoCapture(video_path)
    cropped_faces = []

    # We keep reading until we get `max_frames` valid faces OR the video ends
    while len(cropped_faces) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(frame_rgb)

        # Detect face
        boxes, _ = mtcnn.detect(pil_frame)

        if boxes is not None:
            box = boxes[0] # Take the primary face
            x1, y1, x2, y2 = [int(b) for b in box]

            # Ensure coordinates are within image bounds
            x1 = max(0, x1 - 30)
            y1 = max(0, y1 - 30)
            x2 = min(pil_frame.width, x2 + 30)
            y2 = min(pil_frame.height, y2 + 30)

            # Crop and append
            face_img = pil_frame.crop((x1, y1, x2, y2))
            cropped_faces.append(face_img)

    cap.release()

    # If no faces were found at all, return None
    if len(cropped_faces) == 0:
        return None

    # Process the cropped faces through Swin

    batch = torch.stack([transform(f) for f in cropped_faces]).to(device)


    with torch.no_grad():
        features = swin_tiny_model(batch).cpu() # Shape: (Num_Valid_Faces, 768)

    return features

Extract Features for Entire Dataset

In [115]:
# ==========================================
# MAIN EXTRACTION LOOP
# ==========================================
if __name__ == "__main__":
    # Update this to your actual FaceForensics++ root directory!
    dataset_path = "/content/drive/MyDrive/final year project/fine tuner"

    all_features = {}

    # Walk through the dataset
    print(f"Scanning directory: {dataset_path}")

    # Get all video files first for a cleaner tqdm progress bar
    video_files = []
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.endswith((".mp4", ".avi", ".mov")):
                video_files.append(os.path.join(root, file))

    print(f"Found {len(video_files)} videos. Starting extraction...")

    for video_path in tqdm(video_files):
        feats = get_video_features_mtcnn(video_path)

        if feats is not None:
            # Use relative path as the unique dictionary key
            unique_key = os.path.relpath(video_path, dataset_path)
            all_features[unique_key] = feats

    # Save the new robust features!
    output_filename = "ffplus_swin_features_mtcnn.pt"
    torch.save(all_features, output_filename)
    print(f"\nExtraction complete! Saved to {output_filename}")

Scanning directory: /content/drive/MyDrive/final year project/fine tuner
Found 5 videos. Starting extraction...


100%|██████████| 5/5 [00:15<00:00,  3.16s/it]


Extraction complete! Saved to ffplus_swin_features_mtcnn.pt


In [117]:
#features1 = torch.load(
 #   "/content/drive/MyDrive/final year project/Features 1.pt"
#)
#features2 = torch.load(
#    "/content/drive/MyDrive/final year project/features_of_celeb_v2.pt"
#)
finutuner_feature = torch.load("/content/ffplus_swin_features_mtcnn.pt")


In [46]:
import torch
import os

def merge_datasets(ff_plus_path, custom_reals_path, output_path):
    """
    Merges the massive FaceForensics++ feature dictionary with your new
    custom real videos to anchor the domain.
    """
    print("Loading FaceForensics++ Features...")
    ff_features = ff_plus_path

    print("Loading Custom Real Videos...")
    custom_features = custom_reals_path

    # Check how many we have
    print(f"Initial FF++ Videos: {len(ff_features)}")
    print(f"New Custom Real Videos: {len(custom_features)}")

    # Merge them into a new dictionary
    merged_features = ff_features.copy()

    # Add a unique prefix so the dictionary keys don't collide
    # We will assume all custom videos are REAL (label 0)
    for key, tensor_data in custom_features.items():
        new_key = f"celebdf{key}"
        merged_features[new_key] = tensor_data

    print(f"Total Merged Videos: {len(merged_features)}")

    # Save the new master dataset
    torch.save(merged_features, output_path)
    print(f"Successfully saved master dataset to: {output_path}")

# =========================================================
# HOW TO UPDATE YOUR LABEL ASSIGNMENT (in your training script)
# =========================================================
"""
When you initialize your VideoFeatureDataset, you need to update
the labeling logic to ensure the new custom videos are labeled as REAL (0).

Add this to your __init__ label assignment loop:

if "custom_real" in path_lower:
    self._labels[vid_id] = 0  # Anchor these as REAL
elif "original" in path_lower or "youtube" in path_lower:
    self._labels[vid_id] = 0
elif "manipulated" in path_lower or "fake" in path_lower or "swap" in path_lower:
    self._labels[vid_id] = 1
"""

if __name__ == "__main__":
    # Example Usage:
    merge_datasets(
         ff_plus_path=features1,
         custom_reals_path=features2,
         output_path="master_training_features.pt"
     )
    pass

Loading FaceForensics++ Features...
Loading Custom Real Videos...
Initial FF++ Videos: 7000
New Custom Real Videos: 6529
Total Merged Videos: 13529
Successfully saved master dataset to: master_training_features.pt


In [47]:
features = torch.load("/content/master_training_features.pt")

In [107]:
print(f"Number of videos: {len(finutuner_feature)}")

# To see the shape of one of the feature tensors:
if finutuner_feature:
    first_key = list(finutuner_feature.keys())[0]
    print(f"Shape of features for '{first_key}': {finutuner_feature[first_key].shape}")

Number of videos: 3
Shape of features for 'fake.mp4': torch.Size([32, 768])


# Clasifier Transformer encoder

transformer encoder model



In [49]:
class TemporalTransformerEncoder2(nn.Module):
    def __init__(self,
                 feature_dim=768,      # Swin output dim
                 num_frames=32,
                 num_heads=8,
                 num_layers=4,         # usually 2-6 is enough
                 mlp_ratio=4,
                 num_classes=2,
                 dropout=0.1):
        super().__init__()

        self.num_frames = num_frames
        self.feature_dim = feature_dim

        # Learnable temporal positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1, num_frames, feature_dim))

        # Optional CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, feature_dim))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim,
            nhead=num_heads,
            dim_feedforward=feature_dim * mlp_ratio,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(feature_dim)
        self.head = nn.Linear(feature_dim, num_classes)

    def forward(self, x, frame_indices = None):
        # x shape: (B, T, D)   where T = num_frames, D = feature_dim
        B, T, D = x.shape

        if frame_indices is not None:
            # frame_indices shape must be (B, T)
            # We need to gather the correct positional embeddings for each item in the batch.

            # Expand pos_embed to match batch size: (B, max_frames, D)
            expanded_pos_embed = self.pos_embed.expand(B, -1, -1)

            # Expand indices to match feature dimension: (B, T, D)
            # This allows us to use torch.gather to pluck out the correct 768-dim vectors
            gather_indices = frame_indices.unsqueeze(-1).expand(-1, -1, D)

            # Gather the selected embeddings
            selected_pos_embed = torch.gather(expanded_pos_embed, 1, gather_indices)

            x = x + selected_pos_embed
        else:
            x = x + self.pos_embed[:, :T]

        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)   # (B, T+1, D)

        # Pass through Transformer
        x = self.transformer(x)
        x = self.norm(x)

        # Use CLS token for classification
        cls_output = x[:, 0]
        logits = self.head(cls_output)

        return logits

In [50]:
model = TemporalTransformerEncoder2(
    feature_dim=768,
    num_frames=32,
    num_heads=8,
    num_layers=4
)
model.to(device)


# Dummy data test
batch_size = 2
seq_len = 5 # Let's say the RL agent selected 5 frames
feat_dim = 768

dummy_x = torch.randn(batch_size, seq_len, feat_dim).to(device)

# Simulate RL agent picking different frames for different videos in the batch
dummy_indices = torch.tensor([
        [0, 5, 10, 15, 20],  # Video 1 frame choices
        [2, 4, 8,  16, 31]   # Video 2 frame choices
    ]).to(device)

print("Testing forward pass with frame indices...")
logits = model(dummy_x, frame_indices=dummy_indices)
print("Logits shape:", logits.shape) # Should be (2, 2)
print("Success! The encoder is ready for training.")

'''# Prepare a single video's features for input
if features:
    first_video_key = list(features.keys())[0]
    # Get the feature tensor for the first video
    input_features = features[first_video_key]
    # Add a batch dimension (model expects B, T, D)
    input_features = input_features.unsqueeze(0)
    # Move the input tensor to the same device as the model
    input_features = input_features.to(device)
    logits = model(input_features)   # (B, 2)
    print(logits)
else:
    print("No features found in the dictionary to pass to the model.")'''

Testing forward pass with frame indices...
Logits shape: torch.Size([2, 2])
Success! The encoder is ready for training.


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


'# Prepare a single video\'s features for input\nif features:\n    first_video_key = list(features.keys())[0]\n    # Get the feature tensor for the first video\n    input_features = features[first_video_key]\n    # Add a batch dimension (model expects B, T, D)\n    input_features = input_features.unsqueeze(0)\n    # Move the input tensor to the same device as the model\n    input_features = input_features.to(device)\n    logits = model(input_features)   # (B, 2)\n    print(logits)\nelse:\n    print("No features found in the dictionary to pass to the model.")'

## Training the model

In [51]:
# 1. Initialize Model, Loss, and Optimizer
model = TemporalTransformerEncoder2(feature_dim=768, num_frames=32, num_classes=2).to(device)

# CrossEntropyLoss is standard for multi-class/binary classification with logits
criterion = nn.CrossEntropyLoss()

# AdamW is highly recommended for Transformers over standard Adam
optimizer = optim.AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2) # Use model2's parameters

# Learning rate scheduler (warmup is crucial for Transformers)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Custom Dataset for your features
class VideoFeatureDataset(Dataset):
    def __init__(self, features_dict, num_classes=2):
        self.video_ids = list(features_dict.keys())
        self.features = features_dict
        self._labels = {}

        for vid_id in self.video_ids:
            path_lower = vid_id.lower()
            # CELEB-DF LOGIC
            if "celeb-synthesis" in path_lower:
                self._labels[vid_id] = 1 # Fake
            elif "celeb-real" in path_lower or "youtube-real" in path_lower:
                self._labels[vid_id] = 0 # Real

            # FACEFORENSICS++ LOGIC
            elif "original" in path_lower or "youtube" in path_lower or "real" in path_lower:
                self._labels[vid_id] = 0 # Real
            elif "manipulated" in path_lower or "fake" in path_lower or "swap" in path_lower or "face2face" in path_lower or "neuraltextures" in path_lower:
                self._labels[vid_id] = 1 # Fake
            else:
                # Fallback just in case, but print a warning so you know
                print(f"Warning: Could not determine label for {vid_id}. Defaulting to 1.")
                self._labels[vid_id] = 1


    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]
        feature_tensor = self.features[video_id]
        label = self._labels[video_id]
        return feature_tensor, torch.tensor(label, dtype=torch.long)

# Custom collate_fn for DataLoader to handle varying sequence lengths
def custom_collate_fn(batch):
    # batch is a list of (feature_tensor, label) tuples
    features_list = [item[0] for item in batch]
    labels_list = [item[1] for item in batch]

    # Pad the feature sequences to the maximum length in the current batch
    # pad_sequence expects a list of Tensors, each of shape (L, *), where L is the length of a sequence
    # Here, features_list contains tensors of shape (num_frames, feature_dim)
    padded_features = pad_sequence(features_list, batch_first=True, padding_value=0)

    # Stack the labels
    labels = torch.stack(labels_list)

    return padded_features, labels

def train_transformer(model, dataloader, epochs=10):
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        correct_preds = 0
        total_samples = 0

        for batch_features, batch_labels in dataloader:
            # Move batch to device
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            # batch_features shape: (Batch, Max_Total_Video_Frames_in_Batch, Feature_Dim)
            B, total_frames, D = batch_features.shape

            # --- SIMULATING THE RL AGENT FOR PRE-TRAINING ---
            # Randomly select between 4 and 16 frames to train robustness
            # Ensure num_selected_frames is not greater than total_frames
            max_frames_to_select = min(total_frames, 16)
            if max_frames_to_select < 4:
                # If total_frames is less than 4, select all available frames
                num_selected_frames = total_frames
                # If total_frames is 0, this will be 0, which is handled below
            else:
                num_selected_frames = torch.randint(16,31, (1,)).item()

            # Handle cases where num_selected_frames might be 0 due to very short videos
            if num_selected_frames == 0:
                # Skip this batch if no frames can be selected (e.g., all videos were empty or too short)
                continue

            # Generate random sorted indices for the frames we are "keeping"
            # Shape: (Batch, num_selected_frames)
            frame_indices_list = []
            selected_features_list = []
            for i in range(B):
                # Ensure we don't try to select more frames than available for this specific video
                current_video_frames = batch_features[i].sum(dim=-1).nonzero(as_tuple=True)[0].shape[0] # Count non-padded frames
                actual_frames_to_select = min(num_selected_frames, current_video_frames)

                if actual_frames_to_select == 0:
                    # If no actual frames, add dummy tensors to maintain batch size for stacking
                    frame_indices_list.append(torch.zeros(num_selected_frames, dtype=torch.long, device=device))
                    selected_features_list.append(torch.zeros(num_selected_frames, D, device=device))
                    continue

                perm = torch.randperm(current_video_frames, device=device)[:actual_frames_to_select]
                indices = torch.sort(perm)[0]

                # Pad indices to num_selected_frames if actual_frames_to_select is smaller
                if indices.shape[0] < num_selected_frames:
                    padding_indices = torch.full((num_selected_frames - indices.shape[0],), 0, dtype=torch.long, device=device) # Pad with 0s
                    indices = torch.cat((indices, padding_indices))

                frame_indices_list.append(indices.to(device))
                selected_features_list.append(batch_features[i, indices].to(device))

            frame_indices = torch.stack(frame_indices_list)
            selected_features = torch.stack(selected_features_list)

            # --- FORWARD PASS ---
            optimizer.zero_grad()

            # Pass only selected features for TemporalTransformerEncoder2
            logits = model(selected_features, frame_indices = frame_indices) # TemporalTransformerEncoder2 does NOT use frame_indices

            # Calculate Loss
            loss = criterion(logits, batch_labels)

            # --- BACKWARD PASS ---
            loss.backward()

            # Gradient clipping (prevents exploding gradients in Transformers)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Metrics tracking
            total_loss += loss.item()
            predictions = torch.argmax(logits, dim=1)
            correct_preds += (predictions == batch_labels).sum().item()
            total_samples += B

        scheduler.step()

        epoch_acc = (correct_preds / total_samples) * 100
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f} | Accuracy: {epoch_acc:.2f}%")

def evaluate_model(model, dataloader):
    model.eval() # Set model to evaluation mode
    total_loss = 0
    correct_preds = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for batch_features, batch_labels in dataloader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            logits = model(batch_features) # Pass all features for evaluation

            loss = criterion(logits, batch_labels)
            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)
            correct_preds += (predictions == batch_labels).sum().item()
            total_samples += batch_labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = (correct_preds / total_samples) * 100 if total_samples > 0 else 0
    print(f"\nValidation | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")




In [52]:
# Split features into training and validation sets
video_ids = list(features.keys())
train_video_ids, val_video_ids = train_test_split(video_ids, test_size=0.2, random_state=42)

train_features_dict = {vid_id: features[vid_id] for vid_id in train_video_ids}
val_features_dict = {vid_id: features[vid_id] for vid_id in val_video_ids}


In [118]:
# Split features into training and validation sets
video_ids = list(finutuner_feature.keys())
train_video_ids = video_ids

train_features_dict = {vid_id: finutuner_feature[vid_id] for vid_id in train_video_ids}
train_video_dataset = VideoFeatureDataset(train_features_dict)
train_dataloader_ft = DataLoader(train_video_dataset, batch_size=1 ,shuffle=True, collate_fn=custom_collate_fn)



In [54]:
# Run this once on your training features
import torch
all_feats = torch.cat([v for v in features.values()], dim=0) # Shape: (N, 768)

mean = all_feats.mean(dim=0)
std = all_feats.std(dim=0)

torch.save({'mean': mean, 'std': std}, "dataset_stats.pt")

In [55]:
# Create instances dataset
train_video_dataset = VideoFeatureDataset(train_features_dict)
val_video_dataset = VideoFeatureDataset(val_features_dict)

# ---------------------------------------------------------
# HOW TO FIX YOUR LOSS FUNCTION (In Phase 1 Training)
# ---------------------------------------------------------

# 1. First, count how many Real and Fake videos you actually have
num_real = sum(1 for label in train_video_dataset._labels.values() if label == 0)
num_fake = sum(1 for label in train_video_dataset._labels.values() if label == 1)
total_videos = num_real + num_fake

print(f"Dataset Distribution - REAL: {num_real}, FAKE: {num_fake}")

# 2. Calculate the weights (Inverse Frequency)
# The rarer class (Real) gets a higher weight.
weight_for_real = total_videos / (2.0 * num_real)
weight_for_fake = total_videos / (2.0 * num_fake)

# 3. Create the weight tensor and send it to your GPU/CPU
class_weights = torch.tensor([weight_for_real, weight_for_fake], dtype=torch.float).to(device)

print(f"Class Weights - Real(0): {weight_for_real:.2f}, Fake(1): {weight_for_fake:.2f}")

# 4. Pass the weights to the CrossEntropyLoss
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ---------------------------------------------------------

# Create DataLoaders, applying the custom collate_fn
batch_size = 40
train_dataloader = DataLoader(train_video_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = DataLoader(val_video_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)


Dataset Distribution - REAL: 1498, FAKE: 9325
Class Weights - Real(0): 3.61, Fake(1): 0.58


In [108]:
# Train the model
print("Starting Training...")
train_transformer(model, train_dataloader, epochs=30)

# Evaluate the model on the validation set
print("\nStarting Validation...")
evaluate_model(model, val_dataloader)

Starting Training...


ValueError: too many values to unpack (expected 2)

In [119]:
# Train the model
print("Starting Training...")
train_transformer(model, train_dataloader_ft, epochs=5)


Starting Training...
Epoch 1/5 | Loss: 1.5399 | Accuracy: 60.00%
Epoch 2/5 | Loss: 0.9038 | Accuracy: 60.00%
Epoch 3/5 | Loss: 0.2294 | Accuracy: 100.00%
Epoch 4/5 | Loss: 0.1342 | Accuracy: 100.00%
Epoch 5/5 | Loss: 0.0117 | Accuracy: 100.00%


In [122]:
model_save_path = "/content/drive/MyDrive/final year project/trained_transformer_model_2ds_02.pt"
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

Model weights saved to /content/drive/MyDrive/final year project/trained_transformer_model_2ds_02.pt


In [ ]:
model = TemporalTransformerEncoder2(
    feature_dim=768,
    num_frames=32,
    num_heads=8,
    num_layers=4,
    num_classes=2
).to(device)

# Load the state dictionary into the instantiated model
model.load_state_dict(torch.load("/content/drive/MyDrive/final year project/trained_transformer_model_2ds_01.pt"))

# Set the model to evaluation mode (important for inference, especially with Dropout/BatchNorm)
model.eval()

print("Model loaded successfully and set to evaluation mode.")

Model loaded successfully and set to evaluation mode.


# Reinforcement Learning Environment

In [57]:
class DeepfakeEnv(gym.Env):
    """
    Custom Environment that follows gymnasium interface.
    The RL agent acts as an adaptive frame selector.
    """
    def __init__(self, features_dict, labels_dict, pretrained_transformer, device, max_frames=32):
        super(DeepfakeEnv, self).__init__()

        self.features_dict = features_dict
        self.video_ids = list(features_dict.keys())
        self.labels_dict = labels_dict
        self.model = pretrained_transformer
        self.model.eval() # MUST be in eval mode
        self.device = device
        self.max_frames = max_frames

        # Hyperparameters for the Reward Function
        self.reward_correct = 10.0
        self.reward_wrong = -10.0
        self.penalty_per_frame = -0.2 # Cost of processing a frame

        # Action Space: 0 (Skip Frame), 1 (Select Frame)
        self.action_space = spaces.Discrete(2)

        # Observation Space: The 768-dimensional feature of the current frame
        # (Gym requires numpy arrays, so we use Box)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(768,), dtype=np.float32)

        # Episode tracking variables
        self.current_video_id = None
        self.current_video_features = None
        self.current_true_label = None
        self.current_frame_idx = 0

        self.selected_indices = []
        self.selected_features = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # 1. Pick a random video for the new episode
        self.current_video_id = random.choice(self.video_ids)
        self.current_video_features = self.features_dict[self.current_video_id]
        self.current_true_label = self.labels_dict[self.current_video_id]

        # 2. Reset tracking variables
        self.current_frame_idx = 0
        self.selected_indices = []
        self.selected_features = []

        # 3. Get the first frame's feature to show the agent
        first_frame = self.current_video_features[self.current_frame_idx].numpy()

        info = {}
        return first_frame, info

    def step(self, action):
        # 1. Process the agent's action for the current frame
        if action == 1:
            self.selected_indices.append(self.current_frame_idx)
            self.selected_features.append(self.current_video_features[self.current_frame_idx])

        # 2. Move to the next frame
        self.current_frame_idx += 1

        # 3. Check if the episode is done (reached the end of the video)
        done = self.current_frame_idx >= len(self.current_video_features) or self.current_frame_idx >= self.max_frames

        reward = 0.0
        info = {}

        if done:
            # --- EPISODE END: CALCULATE THE FINAL REWARD ---

            # Edge Case: The agent skipped EVERY frame. Punish heavily.
            if len(self.selected_features) == 0:
                reward = self.reward_wrong - 5.0
            else:
                # Prepare data for the Transformer
                x_input = torch.stack(self.selected_features).unsqueeze(0).to(self.device) # Shape: (1, T_selected, 768)
                indices_input = torch.tensor(self.selected_indices).unsqueeze(0).to(self.device) # Shape: (1, T_selected)

                # Get the Transformer's prediction
                with torch.no_grad(): # No gradients needed for RL reward generation
                    logits = self.model(x_input, frame_indices=indices_input)
                    predicted_class = torch.argmax(logits, dim=1).item()

                # Calculate the core reward
                if predicted_class == self.current_true_label:
                    reward += self.reward_correct
                else:
                    reward += self.reward_wrong

                # Apply the computational penalty for the number of frames used
                compute_penalty = len(self.selected_indices) * self.penalty_per_frame
                reward += compute_penalty

                info['accuracy'] = 1 if predicted_class == self.current_true_label else 0
                info['frames_used'] = len(self.selected_indices)

            # Create a dummy next state since the episode is over
            next_state = np.zeros(768, dtype=np.float32)

        else:
            # Episode is still ongoing, provide the next frame's features
            next_state = self.current_video_features[self.current_frame_idx].numpy()

        # Gymnasium returns: observation, reward, terminated, truncated, info
        return next_state, reward, done, False, info



In [58]:

# ==========================================
# TRAINING THE PPO AGENT
# ==========================================
if __name__ == "__main__":

    # Create label dictionaries
    train_labels_dict = {}
    for vid_id in train_video_ids:
        path_lower = vid_id.lower()
        # CELEB-DF LOGIC
        if "celeb-synthesis" in path_lower:
                train_labels_dict[vid_id] = 1 # Fake
        elif "celeb-real" in path_lower or "youtube-real" in path_lower:
                train_labels_dict[vid_id] = 0 # Real
       # FACEFORENSICS++ LOGIC
        elif "original" in path_lower or "youtube" in path_lower or "real" in path_lower:
                train_labels_dict[vid_id] = 0 # Real
        elif "manipulated" in path_lower or "fake" in path_lower or "swap" in path_lower or "face2face" in path_lower or "neuraltextures" in path_lower:
                train_labels_dict[vid_id] = 1 # Fake
        else:
            train_labels_dict[vid_id] = 1 # Default to fake if unidentifiable

    val_labels_dict = {}
    for vid_id in val_video_ids:
        path_lower = vid_id.lower()
        # CELEB-DF LOGIC
        if "celeb-synthesis" in path_lower:
            val_labels_dict[vid_id] = 1 # Fake
        elif "celeb-real" in path_lower or "youtube-real" in path_lower:
            val_labels_dict[vid_id] = 0 # Real

        # FACEFORENSICS++ LOGIC
        elif "original" in path_lower or "youtube" in path_lower or "real" in path_lower:
                val_labels_dict[vid_id] = 0 # Real
        elif "manipulated" in path_lower or "fake" in path_lower or "swap" in path_lower or "face2face" in path_lower or "neuraltextures" in path_lower:
                val_labels_dict[vid_id] = 1 # Fake
        else:
            val_labels_dict[vid_id] = 1 # Default to fake if unidentifiable

    print("Initializing Deepfake RL Environment...")
    env = DeepfakeEnv(train_features_dict, train_labels_dict, model, device)

    # Optional: Check if the environment follows Gym rules correctly
    check_env(env)

    print("Starting PPO Training...")
    # Initialize PPO model
    # MlpPolicy is used because state is a 1D feature vector (768,)
    ppo_agent = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-5, n_steps=2048)

    # Train the agent
    ppo_agent.learn(total_timesteps=100000)

    print("Training Complete!")
    # Save the trained agent
    ppo_agent.save("/content/drive/MyDrive/final year project/ppo_deepfake_frame_selector_2de_02")

Initializing Deepfake RL Environment...
Starting PPO Training...
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 32       |
|    ep_rew_mean     | 3.22     |
| time/              |          |
|    fps             | 644      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 32            |
|    ep_rew_mean          | 3.49          |
| time/                   |               |
|    fps                  | 510           |
|    iterations           | 2             |
|    time_elapsed         | 8             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 0.0006714939  |
|    clip_fracti

Loading the PPO agent

In [59]:
ppo_agent = PPO.load("/content/drive/MyDrive/final year project/ppo_deepfake_frame_selector_2de_02.zip", device=device)


Evaluating the trained PPO Model

In [60]:
def evaluate_rl_agent(ppo_agent, env, num_episodes=100):
    """
    Evaluates the trained PPO agent on the environment.
    """
    correct_predictions = 0
    total_frames_used = 0
    total_possible_frames = 0

    print(f"--- Evaluating Agent over {num_episodes} Videos ---")

    for episode in range(num_episodes):
        obs, info = env.reset()
        done = False

        #model will manually track actions to be absolutely foolproof
        frames_kept = 0

        while not done:
            # Set deterministic=False. PPO policies can collapse to a single
            # action (like skipping) if forced to be deterministic too early.
            action, _states = ppo_agent.predict(obs, deterministic=False)

            # Track the agent's action in real-time
            if action == 1:
                frames_kept += 1

            obs, reward, done, truncated, info = env.step(action)

        # Safely extract accuracy (if it skipped everything, it defaults to 0)
        accuracy = info.get('accuracy', 0)

        correct_predictions += accuracy
        total_frames_used += frames_kept
        total_possible_frames += env.max_frames

        #Print the first 5 episodes to see the agent's behavior
        if episode < 5:
            print(f"Video {episode+1}: Accuracy={accuracy}, Frames Used={frames_kept}/{env.max_frames}")

    # Calculate final metrics
    final_accuracy = (correct_predictions / num_episodes) * 100
    avg_frames_used = total_frames_used / num_episodes
    frame_reduction_percent = (1 - (total_frames_used / total_possible_frames)) * 100

    print("\n====================================")
    print("      FINAL EVALUATION RESULTS      ")
    print("====================================")
    print(f"Accuracy:              {final_accuracy:.2f}%")
    print(f"Average Frames Used:   {avg_frames_used:.1f} / {env.max_frames}")
    print(f"Computational Savings: {frame_reduction_percent:.2f}% Frame Reduction")
    print("====================================")

# To run this:
# 1. Create a validation environment using your val_features_dict and val_labels_dict
val_env = DeepfakeEnv(val_features_dict, val_labels_dict, model, device)

# 2. Pass trained agent and the validation environment
evaluate_rl_agent(ppo_agent, val_env, num_episodes=len(val_features_dict))

--- Evaluating Agent over 2706 Videos ---
Video 1: Accuracy=0, Frames Used=14/32
Video 2: Accuracy=1, Frames Used=20/32
Video 3: Accuracy=1, Frames Used=15/32
Video 4: Accuracy=1, Frames Used=20/32
Video 5: Accuracy=1, Frames Used=21/32

      FINAL EVALUATION RESULTS      
Accuracy:              69.84%
Average Frames Used:   17.9 / 32
Computational Savings: 44.16% Frame Reduction


Evaluating the model with a video appart from the Dataset

In [123]:
import cv2
import torch
import numpy as np
from torchvision import transforms
from facenet_pytorch import MTCNN
from PIL import Image


def predict_single_video(video_path, swin_model, rl_agent, transformer_model,mtcnn, device, max_frames=32):

    print(f"\nProcessing Video: {video_path}")

    # Initialize the Face Detector
    # margin=30 ensures to capture the whole head (chin, hair) not just the eyes/nose

    cap = cv2.VideoCapture(video_path)
    cropped_faces = []

    while len(cropped_faces) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(frame_rgb)

        # Detect and crop the face
        face = mtcnn(pil_frame)

        # If a face is found, MTCNN returns a tensor scaled [-1, 1].
        # Need to convert it back to a standard PIL image for the existing transforms,
        # or just apply  normalization directly
        # To keep it compatible with training pipeline, get the bounding box instead
        # and crop it manually using PIL.

        boxes, _ = mtcnn.detect(pil_frame)
        if boxes is not None:
            box = boxes[0] # Take the primary face
            # Expand box slightly based on margin
            x1, y1, x2, y2 = [int(b) for b in box]

            # Ensure coordinates are within image bounds
            x1 = max(0, x1 - 30)
            y1 = max(0, y1 - 30)
            x2 = min(pil_frame.width, x2 + 30)
            y2 = min(pil_frame.height, y2 + 30)

            face_img = pil_frame.crop((x1, y1, x2, y2))
            cropped_faces.append(face_img)

    cap.release()

    if len(cropped_faces) == 0:
        return "Error: Could not detect any faces in the video frames."

    print(f"Successfully extracted and cropped {len(cropped_faces)} faces.")

    #Apply transforms to the cropped faces, not the whole TV frame
    batch = torch.stack([transform(f) for f in cropped_faces]).to(device)

    # Extract features using Swin
    swin_model.eval()
    with torch.no_grad():
        video_features = swin_model(batch).cpu() # Shape: (T, 768)

        # Inside predict_video.py, after getting video_features:
    stats = torch.load("dataset_stats.pt")
    mean = stats['mean'].to(device).cpu() # Move to CPU
    std = stats['std'].to(device).cpu()   # Move to CPU

    # Standardize: (X - mean) / std
    video_features = (video_features - mean) / (std + 1e-6)

    # -----------------------------------------
    # STEP 2: RL Agent Frame Selection
    # -----------------------------------------
    selected_features = []
    selected_indices = []

    for i in range(len(video_features)):
        obs = video_features[i].numpy()

        action, _ = rl_agent.predict(obs, deterministic=True)

        if action == 1:
            selected_features.append(video_features[i])
            selected_indices.append(i)

    # IMPROVED FALLBACK: If the RL agent skips everything, uniformly sample frames
    # This prevents the model from just looking at the first 4 (often static) frames.
    if len(selected_features) == 0:
        print("Agent attempted to skip all frames. Applying uniform sampling fallback.")
        num_frames = len(video_features)
        fallback_len = min(6, num_frames)
        # Generate evenly spaced indices (e.g., [0, 6, 12, 18, 24, 30])
        selected_indices = np.linspace(0, num_frames - 1, fallback_len, dtype=int).tolist()
        selected_features = [video_features[i] for i in selected_indices]

    # -----------------------------------------
    # STEP 3: Final Classification
    # -----------------------------------------
    x_input = torch.stack(selected_features).unsqueeze(0).to(device)
    indices_input = torch.tensor(selected_indices).unsqueeze(0).to(device)

    transformer_model.eval()
    with torch.no_grad():
        logits = transformer_model(x_input, frame_indices=indices_input)

        probabilities = torch.softmax(logits, dim=1)[0]

        real_prob = probabilities[0].item() * 100
        fake_prob = probabilities[1].item() * 100

        prediction = torch.argmax(logits, dim=1).item()

    # -----------------------------------------
    # STEP 4: Output Results
    # -----------------------------------------
    # In predict_video.py, replace your prediction logic with this:
    probabilities = torch.softmax(logits, dim=1)[0]
    real_prob = probabilities[0].item()
    fake_prob = probabilities[1].item()

    if abs(real_prob - fake_prob) < 0.20: # If confidence difference is < 20%
        result_label = "UNCERTAIN / INCONCLUSIVE"
        print("The video quality is too different from my training data to determine.")
    else:
        result_label = "FAKE" if fake_prob > real_prob else "REAL"
    #result_label = "FAKE (Deepfake)" if prediction == 1 else "REAL"
    confidence = fake_prob if prediction == 1 else real_prob

    print("====================================")
    print(f"Prediction:    {result_label}")
    print(f"Confidence:    {confidence:.2f}%")
    print(f"Frames Used:   {len(selected_indices)} out of {len(video_features)} original frames")
    print("====================================")

    return prediction, confidence, len(selected_indices)

# ==========================================
# HOW TO RUN IT
# ==========================================
if __name__ == "__main__":
    # Ensure all your models are loaded and sent to the correct device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Replace this with the path to a raw video file you downloaded from the internet or created
    test_video_path = "/content/20260708_164309.mp4"
    # Load the trained PPO agent
    # Make sure 'ppo_deepfake_frame_selector' is the correct path where you saved it.


    # Run the inference
    predict_single_video(
        video_path=test_video_path,
         swin_model=swin_tiny_model,
         mtcnn = mtcnn,
         rl_agent=ppo_agent, # Pass the loaded PPO agent object
         transformer_model=model,
         device=device
      )


Processing Video: /content/20260708_164309.mp4
Successfully extracted and cropped 32 faces.
Prediction:    REAL
Confidence:    1.00%
Frames Used:   12 out of 32 original frames
